In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np

path = r'/Users/leeseungbo/Desktop/학회 데이터(CSV)/3차 전처리 데이터.csv'
df = pd.read_csv(path, low_memory=False)

print(df.shape)
print(df.columns[:30])


In [ ]:
feat = "사고전차량통행속도"
df[feat] = df[feat].replace([-999, -998, 9999, 999], np.nan)  # 너 데이터에 맞게 조정
df[feat+"_isna"] = df[feat].isna().astype(int)
df[feat] = df[feat].fillna(df[feat].median())
s = df[feat]
lo, hi = s.quantile([0.01, 0.99])
df[feat] = s.clip(lo, hi)

In [ ]:
# 1) injury_any (부상여부)
df["injury_any"] = ((df["사망"].fillna(0) + df["중상"].fillna(0) + df["경상"].fillna(0)) > 0).astype(int)

# 2) sev3 (부상 subset에서만 의미 있음: 사망>중상>경상 우선)
inj = df[df["injury_any"] == 1].copy()

conditions = [
    inj["사망"].fillna(0) > 0,
    (inj["사망"].fillna(0) <= 0) & (inj["중상"].fillna(0) > 0),
    (inj["사망"].fillna(0) <= 0) & (inj["중상"].fillna(0) <= 0) & (inj["경상"].fillna(0) > 0),
]
choices = ["사망", "중상", "경상"]
inj["sev3"] = np.select(conditions, choices, default="경상")  # default는 안전하게 경상

print(df["injury_any"].value_counts())
print(inj["sev3"].value_counts())


In [ ]:
import re

target_cols = ["사망","중상","경상","injury_any","sev3"]

# 누수 의심 패턴(너 데이터에 맞춰 필요하면 조정)
leak_regex = r"(사망자|피해|구급차|인명구조|이송|사고등급|EPDO|sev3|injury_any)"

drop_leak = [c for c in df.columns if re.search(leak_regex, c)]
print("누수의심 drop 후보 수:", len(drop_leak))
# print(drop_leak)

# X 구성용 df
X_base = df.drop(columns=target_cols, errors="ignore").drop(columns=drop_leak, errors="ignore").copy()


In [ ]:
from sklearn.model_selection import train_test_split

# 원-핫
X = pd.get_dummies(X_base, dummy_na=True)

# 결측/무한대 처리
X = X.replace([np.inf, -np.inf], np.nan).fillna(-999)

# Stage1 타깃
y1 = df["injury_any"].astype(int)

X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y1, test_size=0.2, random_state=42, stratify=y1
)


In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

neg, pos = (y_train1==0).sum(), (y_train1==1).sum()
scale_pos_weight = neg / max(pos, 1)

clf1 = xgb.XGBClassifier(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

clf1.fit(X_train1, y_train1)

p1 = clf1.predict_proba(X_test1)[:,1]
print("[Stage1] scale_pos_weight:", scale_pos_weight)
print("[Stage1] AUC   :", round(roc_auc_score(y_test1, p1), 4))
print("[Stage1] PR-AUC:", round(average_precision_score(y_test1, p1), 4))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

# 부상 subset에 해당하는 X / y 만들기
inj_idx = (df["injury_any"] == 1)
X2 = X.loc[df.index[inj_idx]].copy()
y2_str = inj["sev3"].copy()

label_map = {"경상":0, "중상":1, "사망":2}
y2 = y2_str.map(label_map).astype(int)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

clf2 = xgb.XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

clf2.fit(X_train2, y_train2)

pred2 = clf2.predict(X_test2)

print("[Stage2] macro-F1     :", round(f1_score(y_test2, pred2, average="macro"), 4))
print("[Stage2] weighted-F1  :", round(f1_score(y_test2, pred2, average="weighted"), 4))
print("[Stage2] balanced acc :", round(balanced_accuracy_score(y_test2, pred2), 4))
print("\n[Stage2] confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_test2, pred2))
print("\n", classification_report(y_test2, pred2, target_names=["경상","중상","사망"], digits=4))


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report

# 0=경상, 1=중상, 2=사망
y_train = y_train2.copy()
y_test  = y_test2.copy()

# Hier-1 타깃: 경상(0) vs 중증(중상+사망=1)
y1_train = (y_train != 0).astype(int)
y1_test  = (y_test  != 0).astype(int)

# 클래스 불균형 보정
neg, pos = (y1_train==0).sum(), (y1_train==1).sum()
spw = neg / max(pos, 1)

clf_a = xgb.XGBClassifier(
    n_estimators=1200, learning_rate=0.05, max_depth=6,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    random_state=42, scale_pos_weight=spw
)
clf_a.fit(X_train2, y1_train)

# Hier-2 데이터: 중증 subset에서만 (중상=0, 사망=1)
idx_sev_train = (y_train != 0)
Xb_train = X_train2[idx_sev_train]
yb_train = (y_train[idx_sev_train] == 2).astype(int)

idx_sev_test = (y_test != 0)
Xb_test = X_test2[idx_sev_test]
yb_test = (y_test[idx_sev_test] == 2).astype(int)

# Hier-2는 보통 잘 맞음. 필요시 scale_pos_weight는 사망비율로
neg2, pos2 = (yb_train==0).sum(), (yb_train==1).sum()
spw2 = neg2 / max(pos2, 1)

clf_b = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=5,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    random_state=42, scale_pos_weight=spw2
)
clf_b.fit(Xb_train, yb_train)

def predict_hier(X_input, th1=0.5, th2=0.5):
    p1 = clf_a.predict_proba(X_input)[:, 1]
    is_severe = (p1 >= th1)

    pred3 = np.zeros(len(X_input), dtype=int)  # default=경상(0)
    if is_severe.any():
        p2 = clf_b.predict_proba(X_input[is_severe])[:, 1]
        is_fatal = (p2 >= th2)

        pred3[is_severe] = 1  # 중증이면 기본 중상(1)
        pred3[np.where(is_severe)[0][is_fatal]] = 2  # 사망(2)
    return pred3

# th1 스윕: 중상 recall 끌어올리려면 보통 th1을 낮추는 방향
for th1 in [0.30, 0.35, 0.40, 0.45, 0.50]:
    pred = predict_hier(X_test2, th1=th1, th2=0.5)
    macro = f1_score(y_test, pred, average="macro")
    bal  = balanced_accuracy_score(y_test, pred)
    print(f"th1={th1:.2f} macro-F1={macro:.4f} balacc={bal:.4f}")

# 최종 선택 예: th1=0.40
th_best = 0.40
pred_best = predict_hier(X_test2, th1=th_best, th2=0.5)

print("\n[Final Hier] th1=", th_best)
print("macro-F1:", round(f1_score(y_test, pred_best, average="macro"), 4))
print("balacc  :", round(balanced_accuracy_score(y_test, pred_best), 4))
print("CM:\n", confusion_matrix(y_test, pred_best))
print(classification_report(y_test, pred_best, target_names=["경상","중상","사망"], digits=4))


In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, confusion_matrix

def eval_grid(th1_list, th2_list):
    rows = []
    for th1 in th1_list:
        for th2 in th2_list:
            pred = predict_hier(X_test2, th1=th1, th2=th2)
            macro = f1_score(y_test2, pred, average="macro")
            bal = balanced_accuracy_score(y_test2, pred)
            rec_mid = recall_score(y_test2, pred, labels=[1], average=None)[0]  # 중상 recall
            rows.append((th1, th2, macro, bal, rec_mid))
    return sorted(rows, key=lambda x: x[2], reverse=True)

th1_list = [0.40, 0.45, 0.50, 0.55]
th2_list = [0.50, 0.60, 0.70, 0.80]

rows = eval_grid(th1_list, th2_list)
print("Top 10 by macro-F1:")
for r in rows[:10]:
    print(f"th1={r[0]:.2f}, th2={r[1]:.2f}, macro-F1={r[2]:.4f}, balacc={r[3]:.4f}, mid_recall={r[4]:.4f}")


In [ ]:
best = rows[0]
th1_best, th2_best = best[0], best[1]
pred_best = predict_hier(X_test2, th1=th1_best, th2=th2_best)

print("Best:", best)
print(confusion_matrix(y_test2, pred_best))
print(classification_report(y_test2, pred_best, target_names=["경상","중상","사망"], digits=4))


In [ ]:
import numpy as np
import pandas as pd

X_shap = X_test2.copy()

# 1) object/boolean 등 전부 numeric으로 변환 시도
for c in X_shap.columns:
    if X_shap[c].dtype == "object" or str(X_shap[c].dtype).startswith("string"):
        X_shap[c] = pd.to_numeric(X_shap[c], errors="coerce")

# 2) bool도 0/1로
bool_cols = X_shap.select_dtypes(include=["bool"]).columns
X_shap[bool_cols] = X_shap[bool_cols].astype(int)

# 3) inf/NaN 처리
X_shap = X_shap.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4) float로 통일
X_shap = X_shap.astype(np.float32)

print("X_shap dtypes check:", X_shap.dtypes.value_counts().head())

In [ ]:
import shap
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

explainer = shap.TreeExplainer(clf2)

sv = explainer.shap_values(X_shap)

# 멀티클래스: list 또는 (K,n,p) / (n,p,K) 케이스 대응
k = 2  # 사망 클래스
if isinstance(sv, list):
    sv_k = sv[k]                 # (n,p)
else:
    if sv.ndim == 3 and sv.shape[0] == 3:
        sv_k = sv[k]             # (n,p)
    elif sv.ndim == 3 and sv.shape[2] == 3:
        sv_k = sv[:, :, k]       # (n,p)
    else:
        raise ValueError("Unexpected shap_values shape:", sv.shape)

# bias 열 붙는 케이스 제거
if sv_k.shape[1] == X_shap.shape[1] + 1:
    sv_k = sv_k[:, :-1]

print("final shapes:", sv_k.shape, X_shap.shape)

plt.figure()
shap.summary_plot(sv_k, X_shap, show=False, max_display=20)
plt.title("SHAP summary - 사망")
plt.tight_layout()
plt.show()

### SHAP dependence plot

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # macOS
plt.rcParams["axes.unicode_minus"] = False

# SHAP에 넣을 데이터는 "학습에 쓴 것과 동일한 전처리된 X"여야 함
X_shap = X_test2.copy()

# (혹시 object 남아있으면 안전 처리)
X_shap = X_shap.replace([np.inf, -np.inf], np.nan).fillna(0)
X_shap = X_shap.astype(np.float32)

explainer = shap.TreeExplainer(clf2)
sv = explainer.shap_values(X_shap)

k = 2  # 사망 클래스 인덱스
if isinstance(sv, list):
    sv_k = sv[k]
else:
    # (K,n,p) or (n,p,K) 대응
    if sv.ndim == 3 and sv.shape[0] == 3:
        sv_k = sv[k]
    elif sv.ndim == 3 and sv.shape[2] == 3:
        sv_k = sv[:, :, k]
    else:
        raise ValueError("Unexpected shap shape:", sv.shape)

# bias 열(+1) 있으면 제거
if sv_k.shape[1] == X_shap.shape[1] + 1:
    sv_k = sv_k[:, :-1]

print("sv_k:", sv_k.shape, "X_shap:", X_shap.shape)


In [ ]:
feat = "사고전차량통행속도"
inter = "사고유형_차-사람"

plt.figure()
shap.dependence_plot(
    feat, sv_k, X_shap,
    interaction_index=inter,
    show=False
)
plt.title("Dependence: 사고전차량통행속도 × 사고유형_차-사람 (사망 클래스)")
plt.tight_layout()학
out_path = "/Users/leeseungbo/Desktop/dep_speed_x_car_ped.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print("saved:", out_path)
